# ReachGrasp Riemannian Geodesic Primitives

This notebook runs one experiment: represent each ReachGrasp reach-return trial as a sequence of geodesic primitives in right-arm joint space.

The experiment asks whether motion is better reconstructed by primitives that respect the configuration-dependent mass metric

$$
G(q) = M(q)
$$

than by straight-line Euclidean primitives. The notebook keeps only the pieces needed for that comparison: data loading, reach cropping, smoothing, Riemannian segmentation, primitive reconstruction, baseline reconstruction, diagnostics, and plots.


## 1. Imports and Workspace Setup

The notebook uses NumPy, Matplotlib, SciPy-backed functions from `riemannian`, and `MassMetricModel` for $G(q)=M(q)$.


In [ ]:
# --------------------
# Environment and imports
# --------------------
from __future__ import annotations

import csv
import os
import warnings
from dataclasses import dataclass
from pathlib import Path
from motion_primitives.paths import PROJECT_ROOT

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm
from motion_primitives import randg
from motion_primitives.paths import PROJECT_ROOT
from riemannian.extraction import fit_arc_cubic, is_tiny_segment, smooth_boundary_speeds
from riemannian.geodesics import integrate_geodesic, log_map_shooting, straight_line_path
from riemannian.metric import MassMetricModel, Metric, norm, regularize_spd
from riemannian.preprocessing import smooth_and_differentiate
from riemannian.segmentation import (
    first_meaningful_velocity_index, metric_speeds, segment_riemannian,
)



## 2. Experiment Configuration

This is the complete control panel for the joint-space experiment.

The four model keys define the comparison:

1. `riemannian`: segment by Riemannian velocity angle and reconstruct each segment with $G(q)=M(q)$.
2. `euclidean_paper`: segment by simultaneous joint-velocity zero crossings and reconstruct straight lines.
3. `euclidean_controlled`: use the Riemannian segment boundaries but reconstruct straight lines with $G(q)=I$.
4. `constant_metric`: use the Riemannian boundaries and a frozen average metric $G(q)=\bar{M}$, isolating the effect of changing $M(q)$.


In [ ]:
# --------------------
# Configuration
# --------------------
CONFIG = {
    # Data selection.
    "dataset_path": None,
    "subjects": ["sub-01"],#, "sub-02"],
    "tasks": ["FroRea"],#, "Pour", "Cyl"],
    "output_dir": PROJECT_ROOT / "results/ReachGrasp/riemannian/reach_grasp_geodesics",

    # Run switches.
    # This notebook remains a comparison/diagnostic notebook. The human primitive
    # extraction itself is the "riemannian" model.
    "models_to_run": [
        "riemannian",
        "euclidean_paper",
        "euclidean_controlled",
        "constant_metric",
    ],

    # Smoothing and differentiation.
    # Klein et al. use a Savitzky-Golay filter with a 21-sample window for joint
    # angles and analytical derivatives. Keep these explicit so the extraction is reproducible.
    "savgol_window_samples": 31,
    "savgol_polyorder": 2,

    # Riemannian segmentation parameters.
    "delta_theta": np.pi / 3,
    "angle_confirm_samples": 5,
    "min_segment_samples": 10,
    "speed_floor_fraction": 0.05,
    "min_reach_duration": 0.3,
    "max_segments": 15,
    "min_hand_speed": 0.005,

    # Tiny-segment handling. This is a fixed cleanup rule, not a parameter optimization sweep.
    "tiny_segment_policy": "merge",  # one of: "merge", "local", "shoot"
    "min_tiny_segment_duration": 0.15,

    # Paper Euclidean baseline segmentation.
    "euclidean_zero_crossing_window_s": 0.050,
    "euclidean_zero_crossing_min_joints": 3,

    # Metric parameters.
    "metric_eig_floor": 1e-7,
    "metric_finite_difference_step": 1e-5,
    "constant_metric_stride": 5,

    # Geodesic/log-map parameters.
    "geodesic_path_samples": 80,
    "log_map_max_nfev": 60,
    "log_map_endpoint_tol": 2e-3,
    "geodesic_max_step": 0.03,

    # Validation thresholds.
    "max_metric_joint_mse": 0.05,
    "max_velocity_profile_mse": 0.5,
    "max_allowed_energy_drift": 0.05,
    "max_allowed_cond_M": 1e8,

    # Primitive-library export.
    "primitive_library_models": ["Full Riemannian M(q)"],

    # Plotting.
    "plot_first_n_trials": 3,
}

output_full = PROJECT_ROOT / "experiments_Riemannian" / CONFIG["output_dir"]
output_full.mkdir(parents=True, exist_ok=True)
CONFIG["output_dir_full"] = output_full

CONFIG


## 3. Trial representation and dataset loading

A `ReachGraspTrial` is the notebook's plain container for one repetition of one task. Each repetition is processed independently. The minimum required data are:

- `time`: sample times,
- `arm_joint_angles`: the 9 right-arm joint coordinates used by the RBDL model,
- `joint_names`: column names for those coordinates,
- `hand_motion`: ReachGrasp Vicon hand/wrist joint channels used for movement onset and diagnostic plots.

The loader uses `split_repetitions=True`, so every returned repetition becomes one row in the final summary table.


In [ ]:
# --------------------
# Dataset representation
# --------------------
RIGHT_ARM_JOINTS = (
    "RShoulder_X", "RShoulder_Y", "RShoulder_Z",
    "RElbow_X", "RElbow_Y", "RElbow_Z",
    "RWrist_X", "RWrist_Y", "RWrist_Z",
)

RIGHT_HAND_MOTION_JOINTS = (
    "RWrist_X", "RWrist_Y", "RWrist_Z",
    "RIndex_X", "RIndex_Y", "RIndex_Z",
    "RMiddle_X", "RMiddle_Y", "RMiddle_Z",
    "RRing_X", "RRing_Y", "RRing_Z",
    "RPinky_X", "RPinky_Y", "RPinky_Z",
    "RThumb_STT_X", "RThumb_STT_Y", "RThumb_STT_Z",
    "RThumb_MCP_X",
)


@dataclass
class ReachGraspTrial:
    subject: str
    task: str
    trial: int
    time: np.ndarray
    arm_joint_angles: np.ndarray
    joint_names: tuple[str, ...]
    hand_motion: np.ndarray
    hand_motion_names: tuple[str, ...]
    tactile: np.ndarray | None = None
    finger_angles: np.ndarray | None = None
    angles_in_degrees: bool = True


def load_randg_trials(subject: str, task: str, dataset_path: str | Path | None = None) -> list[ReachGraspTrial]:
    if randg is None:
        raise ModuleNotFoundError("motion_primitives.randg is required to load ReachGrasp trials.")

    repetitions = randg.load_arm_joint_data(subject, task, dataset_path=dataset_path, return_all_joints=True, split_repetitions=True)
    trials = []
    for trial_index, (time_vals, joint_angles, _info) in enumerate(repetitions, start=0):
        missing_arm = [name for name in RIGHT_ARM_JOINTS if name not in joint_angles.columns]
        if missing_arm:
            raise ValueError(f"ReachGrasp trial is missing required RBDL arm joints: {missing_arm}")

        available_hand = tuple(name for name in RIGHT_HAND_MOTION_JOINTS if name in joint_angles.columns)
        if len(available_hand) < 3:
            raise ValueError("ReachGrasp trial does not contain enough hand/wrist motion channels for movement onset.")

        q = joint_angles.loc[:, RIGHT_ARM_JOINTS].to_numpy(dtype=float)
        hand_motion = joint_angles.loc[:, available_hand].to_numpy(dtype=float)
        trials.append(ReachGraspTrial(
            subject=subject,
            task=task,
            trial=trial_index,
            time=np.asarray(time_vals, dtype=float),
            arm_joint_angles=q,
            joint_names=RIGHT_ARM_JOINTS,
            hand_motion=hand_motion,
            hand_motion_names=available_hand,
            angles_in_degrees=True,
        ))

    print(f"Loaded {len(trials)} trials for {subject} {task}")
    return trials


## 4. Metric and Inner Products

The Riemannian model uses the arm mass matrix as the metric. At each configuration $q$, the inner product, norm, and angle are

$$
\langle u, v \rangle_q = u^\top M(q)v,
$$

$$
\|u\|_q = \sqrt{u^\top M(q)u},
$$

$$
\theta_q(u,v)=\arccos\left(\frac{\langle u,v\rangle_q}{\|u\|_q\,\|v\|_q}\right).
$$

The constant and Euclidean metrics below are baselines for isolating which part of the method matters.


In [ ]:
# --------------------
# Metric model and Riemannian operations
# --------------------
class EuclideanMetricModel:
    """Identity metric used as a straight-line reconstruction baseline."""

    name = "Euclidean"
    is_constant = True

    def __init__(self, dim: int):
        self.dim = int(dim)
        self._I = np.eye(self.dim)

    def __call__(self, q: np.ndarray) -> np.ndarray:
        return self._I

    def grad(self, q: np.ndarray) -> np.ndarray:
        return np.zeros((self.dim, self.dim, self.dim), dtype=float)

    def evaluate_with_diagnostics(self, q: np.ndarray) -> tuple[np.ndarray, dict]:
        return self._I, {"condition_number": 1.0, "num_regularized_eigenvalues": 0, "max_regularization": 0.0}


class ConstantMetricModel:
    """Frozen mass-matrix baseline. Paths are straight, but lengths and speeds use M0."""

    name = "Constant metric"
    is_constant = True

    def __init__(self, M0: np.ndarray):
        self.M0 = regularize_spd(np.asarray(M0, dtype=float), eig_floor=CONFIG["metric_eig_floor"])
        self.dim = self.M0.shape[0]

    def __call__(self, q: np.ndarray) -> np.ndarray:
        return self.M0

    def grad(self, q: np.ndarray) -> np.ndarray:
        return np.zeros((self.dim, self.dim, self.dim), dtype=float)

    def evaluate_with_diagnostics(self, q: np.ndarray) -> tuple[np.ndarray, dict]:
        return self.M0, {"condition_number": float(np.linalg.cond(self.M0)), "num_regularized_eigenvalues": 0, "max_regularization": 0.0}


def constant_metric_from_trial(q: np.ndarray, G: Metric, stride: int = 5) -> ConstantMetricModel:
    """Average direct RBDL mass matrices along the trial and freeze the result."""
    stride = max(1, int(stride))
    samples = q[::stride]
    if len(samples) == 0:
        samples = q[:1]
    matrices = [G(q_t) for q_t in samples]
    return ConstantMetricModel(np.mean(matrices, axis=0))




In [ ]:
# --------------------
# Euclidean baseline segmentation
# --------------------
def segment_euclidean_zero_velocity_crossing(qdot: np.ndarray, dt: float, window_s: float = 0.050, min_joints: int = 3, min_segment_samples: int = 3) -> list[tuple[int, int]]:
    """
    Segment the paper-style Euclidean baseline.

    A boundary is inserted when at least `min_joints` joint velocities change sign
    inside a short time window. A refractory gap prevents the same crossing event
    from generating many adjacent boundaries.
    """
    qdot = np.asarray(qdot, dtype=float)
    if len(qdot) < 2:
        return []

    half_window = max(1, int(round(window_s / max(dt, 1e-12) / 2.0)))
    min_boundary_gap = max(int(min_segment_samples), half_window)
    boundaries = [0]
    signs = np.sign(qdot)
    signs[signs == 0.0] = np.nan

    for t in range(1, len(qdot)):
        left = max(0, t - half_window)
        right = min(len(qdot), t + half_window + 1)
        crossed = 0
        for j in range(qdot.shape[1]):
            values = signs[left:right, j]
            values = values[np.isfinite(values)]
            if len(values) >= 2 and np.nanmin(values) < 0 and np.nanmax(values) > 0:
                crossed += 1
        if crossed >= min_joints and (t - boundaries[-1]) >= min_boundary_gap:
            boundaries.append(t)

    if boundaries[-1] != len(qdot) - 1:
        boundaries.append(len(qdot) - 1)
    segments = [(boundaries[i], boundaries[i + 1]) for i in range(len(boundaries) - 1) if boundaries[i + 1] > boundaries[i]]
    return segments or [(0, len(qdot) - 1)]


## 5. Reach-Return Crop

The trial is cropped from movement onset to the recorded end of the repetition. Movement onset is estimated from the hand/wrist motion, but all reconstructions are evaluated in right-arm joint space.


In [ ]:
# --------------------
# Reach-return action crop
# --------------------
def finite_difference_velocity(x: np.ndarray, dt: float) -> np.ndarray:
    velocity = np.zeros_like(x, dtype=float)
    if len(x) < 2:
        return velocity

    velocity[1:-1] = (x[2:] - x[:-2]) / (2.0 * dt)
    velocity[0] = (x[1] - x[0]) / dt
    velocity[-1] = (x[-1] - x[-2]) / dt
    return velocity


def detect_movement_onset(motion_signal: np.ndarray, dt: float, frac: float = 0.05, sustain: int = 5) -> int:
    speed = np.linalg.norm(finite_difference_velocity(motion_signal, dt), axis=1)
    if len(speed) == 0 or float(np.max(speed)) <= 0.0:
        return 0

    threshold = frac * float(np.max(speed))
    sustain = max(1, int(sustain))
    for t in range(0, max(len(speed) - sustain + 1, 0)):
        if np.all(speed[t:t + sustain] > threshold):
            return t

    return 0


## 6. Smooth Joint Motion and Segment by Riemannian Direction

For each cropped trial, the method estimates smooth joint positions $q(t)$ and velocities $\dot{q}(t)$.

Riemannian segmentation then proceeds top-down:

1. Choose the first meaningful velocity as the reference direction $v_0$.
2. Parallel-transport that reference direction along the observed path.
3. At each sample $t$, compute the Riemannian angle $\theta_t$ between the transported reference velocity $\tilde{v}_0(t)$ and the current velocity $\dot{q}(t)$:

$$
\theta_t = \theta_{q(t)}\left(\tilde{v}_0(t), \dot{q}(t)\right).
$$

4. Start a new segment when

$$
\theta_t > \Delta\theta
$$

and the current segment has enough samples.

This gives primitives whose boundaries are defined by changes in motion direction under $M(q)$, not by ordinary Euclidean angle.


## 7. Build Geodesic Primitives and Baselines

Each segment is represented by a spatial path plus a temporal profile.

For the full Riemannian model, the notebook solves a shooting problem: find the initial velocity $v_0$ such that the geodesic starting at $q_i$ reaches $q_f$ at unit phase:

$$
q(0)=q_i, \qquad q(1)=q_f.
$$

The geodesic is integrated as the first-order system

$$
\frac{dq}{ds}=v,
$$

$$
\frac{dv}{ds}=-M(q)^{-1}\Gamma(q)[v,v].
$$

The Euclidean baselines use straight paths between the same endpoint pairs:

$$
q(s) = (1-s)q_i + s q_f, \qquad s \in [0,1].
$$

All models reuse the measured segment timing so the comparison focuses on spatial primitive geometry and segmentation.


In [ ]:
# --------------------
# Geodesic primitives, baselines, diagnostics, and temporal reconstruction
# --------------------
def metric_with_diagnostics(G: Metric, q: np.ndarray) -> tuple[np.ndarray, dict]:
    if hasattr(G, "evaluate_with_diagnostics"):
        Gq, info = G.evaluate_with_diagnostics(q)
    else:
        Gq = G(q)
        info = {"condition_number": float(np.linalg.cond(Gq)), "num_regularized_eigenvalues": 0, "max_regularization": 0.0}
    info = dict(info)
    info.setdefault("condition_number", float(np.linalg.cond(Gq)))
    info.setdefault("num_regularized_eigenvalues", 0)
    info.setdefault("max_regularization", 0.0)
    return Gq, info


def euclidean_path_length(q_path: np.ndarray) -> float:
    q_path = np.asarray(q_path, dtype=float)
    if len(q_path) < 2:
        return 0.0
    return float(np.sum(np.linalg.norm(np.diff(q_path, axis=0), axis=1)))


def cumulative_riemannian_lengths(q_path: np.ndarray, G: Metric) -> np.ndarray:
    q_path = np.asarray(q_path, dtype=float)
    cumulative = np.zeros(len(q_path), dtype=float)
    for m in range(len(q_path) - 1):
        dq = q_path[m + 1] - q_path[m]
        q_mid = 0.5 * (q_path[m] + q_path[m + 1])
        cumulative[m + 1] = cumulative[m] + float(np.sqrt(max(dq.T @ G(q_mid) @ dq, 0.0)))
    return cumulative


def local_geodesic_approximation(q_i: np.ndarray, q_f: np.ndarray, G: Metric, path_samples: int,
                                 max_step: float, endpoint_tol: float) -> tuple[np.ndarray, np.ndarray, np.ndarray, dict]:
    initial = np.asarray(q_f, dtype=float) - np.asarray(q_i, dtype=float)
    q_path, v_path = integrate_geodesic(q_i, initial, G, samples=path_samples, max_step=max_step)
    endpoint_error = float(np.linalg.norm(q_path[-1] - q_f))
    info = {
        "method": "local_initial_no_shooting",
        "success": bool(endpoint_error <= endpoint_tol),
        "endpoint_error": endpoint_error,
        "nfev": 0,
        "integrator": "solve_ivp",
        "approximate": True,
    }
    return initial, q_path, v_path, info

def neighbor_alignment(q: np.ndarray, tiny: tuple[int, int], neighbor: tuple[int, int]) -> float:
    tiny_vec = q[tiny[1]] - q[tiny[0]]
    neighbor_vec = q[neighbor[1]] - q[neighbor[0]]
    denom = np.linalg.norm(tiny_vec) * np.linalg.norm(neighbor_vec)
    if denom < 1e-12:
        return -1.0
    return float(abs(np.dot(tiny_vec, neighbor_vec) / denom))


def handle_tiny_segments(segments: list[tuple[int, int]], q: np.ndarray, dt: float, policy: str, min_duration: float) -> tuple[list[tuple[int, int]], set[tuple[int, int]], dict, dict]:
    policy = str(policy).lower()
    if policy not in {"merge", "local", "shoot"}:
        raise ValueError("tiny_segment_policy must be 'merge', 'local', or 'shoot'.")
    t = np.arange(len(q), dtype=float) * dt
    tiny_original = {segment for segment in segments if is_tiny_segment(segment, t, min_duration)}
    counts = {"tiny_segments_detected": len(tiny_original), "tiny_segments_merged": 0, "tiny_segments_local": 0, "tiny_segments_shot": 0}
    status = {segment: ("tiny_shot" if segment in tiny_original and policy == "shoot" else "normal") for segment in segments}
    if policy == "shoot":
        counts["tiny_segments_shot"] = len(tiny_original)
        return segments, set(), counts, status
    if policy == "local":
        for segment in tiny_original:
            status[segment] = "local_approximation"
        counts["tiny_segments_local"] = len(tiny_original)
        return segments, set(tiny_original), counts, status
    merged = list(segments)
    merged_status = {segment: ("tiny" if segment in tiny_original else "normal") for segment in merged}
    while True:
        tiny_index = None
        for i, segment in enumerate(merged):
            if is_tiny_segment(segment, t, min_duration) and len(merged) > 1:
                tiny_index = i
                break
        if tiny_index is None:
            break
        segment = merged[tiny_index]
        prev_score = neighbor_alignment(q, segment, merged[tiny_index - 1]) if tiny_index > 0 else -1.0
        next_score = neighbor_alignment(q, segment, merged[tiny_index + 1]) if tiny_index + 1 < len(merged) else -1.0
        if prev_score >= next_score and tiny_index > 0:
            new_segment = (merged[tiny_index - 1][0], segment[1])
            del merged[tiny_index]
            merged[tiny_index - 1] = new_segment
        elif tiny_index + 1 < len(merged):
            new_segment = (segment[0], merged[tiny_index + 1][1])
            del merged[tiny_index + 1]
            merged[tiny_index] = new_segment
        else:
            break
        merged_status[new_segment] = "merged_tiny"
        counts["tiny_segments_merged"] += 1
    return merged, set(), counts, {segment: merged_status.get(segment, "normal") for segment in merged}


def build_primitives_riemannian(q: np.ndarray, segments: list[tuple[int, int]], G: Metric, path_samples: int = 80, log_map_max_nfev: int = 35, endpoint_tol: float = 2e-3, max_step: float = 0.03, local_segments: set[tuple[int, int]] | None = None, segment_status: dict | None = None) -> list[dict]:
    primitives = []
    local_segments = local_segments or set()
    segment_status = segment_status or {}
    for start, end in tqdm(segments, desc="Riemannian primitives", unit="primitive"):
        if end <= start:
            continue
        q_i = q[start].copy()
        q_f = q[end].copy()
        if (start, end) in local_segments:
            v_i, q_path, v_path, info = local_geodesic_approximation(q_i, q_f, G, path_samples, max_step, endpoint_tol)
        else:
            v_i, q_path, v_path, info = log_map_shooting(
                q_i, q_f, G, samples=path_samples, max_nfev=log_map_max_nfev,
                endpoint_tol=endpoint_tol, max_step=max_step)
        arc_lengths = cumulative_riemannian_lengths(q_path, G) if len(q_path) else np.array([0.0])
        primitives.append({"start": int(start), "end": int(end), "q_i": q_i, "q_f": q_f, "v_i": v_i, "q_path": q_path, "v_path": v_path, "arc_lengths": arc_lengths, "length": float(arc_lengths[-1]) if len(arc_lengths) else 0.0, "log_map_info": info, "tiny_segment_status": segment_status.get((start, end), "normal")})
    return primitives

def build_primitives_straight(q: np.ndarray, segments: list[tuple[int, int]], G: Metric, path_samples: int = 80, segment_status: dict | None = None) -> list[dict]:
    primitives = []
    segment_status = segment_status or {}
    for start, end in segments:
        if end <= start:
            continue
        q_i = q[start].copy()
        q_f = q[end].copy()
        q_path, v_path = straight_line_path(q_i, q_f, path_samples)
        arc_lengths = cumulative_riemannian_lengths(q_path, G)
        primitives.append({"start": int(start), "end": int(end), "q_i": q_i, "q_f": q_f, "v_i": q_f - q_i, "q_path": q_path, "v_path": v_path, "arc_lengths": arc_lengths, "length": float(arc_lengths[-1]) if len(arc_lengths) else 0.0, "log_map_info": {"method": "straight_line", "success": True, "endpoint_error": 0.0, "nfev": 0, "integrator": "closed_form", "approximate": False}, "tiny_segment_status": segment_status.get((start, end), "normal")})
    return primitives


def arc_phase(coeffs: np.ndarray, t: float) -> float:
    a0, a1, a2, a3 = coeffs
    return float(a0 + a1 * t + a2 * t**2 + a3 * t**3)


def add_temporal_profiles(primitives: list[dict], q: np.ndarray, qdot: np.ndarray, dt: float, G: Metric) -> list[dict]:
    for P in primitives:
        start = P["start"]
        end = P["end"]
        duration = max((end - start) * dt, dt)
        length = max(float(P["length"]), 1e-8)
        speed0 = norm(q[start], qdot[start], G)
        speedf = norm(q[end], qdot[end], G)
        P["duration"] = float(duration)
        P["speed0"] = float(speed0)
        P["speedf"] = float(speedf)
        P["arc_phase_coeffs"] = fit_arc_cubic(duration, length, speed0, speedf)
    return primitives


def interpolate_path_by_arc_length(P: dict, s_arc: float) -> np.ndarray:
    q_path = P["q_path"]
    arc = P["arc_lengths"]
    if len(q_path) == 0:
        return P["q_i"].copy()
    if len(q_path) == 1 or arc[-1] <= 1e-12:
        return q_path[-1].copy()
    s_arc = float(np.clip(s_arc, 0.0, arc[-1]))
    out = np.empty(q_path.shape[1], dtype=float)
    for j in range(q_path.shape[1]):
        out[j] = np.interp(s_arc, arc, q_path[:, j])
    return out


def reconstruct_segment_riemannian(P: dict, dt: float) -> np.ndarray:
    n = P["end"] - P["start"] + 1
    ts = np.linspace(0.0, P["duration"], n)
    return np.asarray([interpolate_path_by_arc_length(P, arc_phase(P["arc_phase_coeffs"], float(tau))) for tau in ts])


def reconstruct_all_riemannian(primitives: list[dict], target_length: int, dt: float) -> np.ndarray:
    """Concatenate reconstructed primitive trajectories to a fixed target length."""
    if np.ndim(target_length) != 0:
        raise TypeError(f"target_length must be a scalar integer, got shape {np.shape(target_length)}. Did you pass q instead of len(q)?")
    if np.ndim(dt) != 0:
        raise TypeError(f"dt must be a scalar float, got shape {np.shape(dt)}.")

    target_length = int(target_length)
    dt = float(dt)
    if target_length <= 0:
        return np.empty((0, 0))

    q_hat_segments = []
    for P in primitives:
        q_seg = reconstruct_segment_riemannian(P, dt)
        if q_hat_segments and len(q_seg) > 0 and np.linalg.norm(q_hat_segments[-1][-1] - q_seg[0]) < 1e-9:
            q_seg = q_seg[1:]
        if len(q_seg) > 0:
            q_hat_segments.append(q_seg)
    if not q_hat_segments:
        return np.empty((0, 0))
    q_hat = np.vstack(q_hat_segments)
    if len(q_hat) > target_length:
        return q_hat[:target_length]
    if len(q_hat) < target_length:
        return np.vstack([q_hat, np.repeat(q_hat[-1:], target_length - len(q_hat), axis=0)])
    return q_hat


def primitive_diagnostics(P: dict, G: Metric, dt: float, model_name: str, segment_index: int, max_allowed_energy_drift: float, max_allowed_cond_M: float) -> dict:
    q_path = np.asarray(P.get("q_path", []), dtype=float)
    v_path = np.asarray(P.get("v_path", []), dtype=float)
    info = P.get("log_map_info", {})
    energies = []
    conds = []
    num_regularized = 0
    max_regularization = 0.0
    if len(q_path) and len(v_path):
        for q_s, v_s in zip(q_path, v_path):
            Gq, metric_info = metric_with_diagnostics(G, q_s)
            energies.append(float(0.5 * v_s.T @ Gq @ v_s))
            conds.append(float(metric_info.get("condition_number", np.linalg.cond(Gq))))
            num_regularized += int(metric_info.get("num_regularized_eigenvalues", 0) > 0)
            max_regularization = max(max_regularization, float(metric_info.get("max_regularization", 0.0)))
    energies = np.asarray(energies, dtype=float)
    energy_drift = float((np.max(energies) - np.min(energies)) / (np.mean(energies) + 1e-12)) if len(energies) and np.all(np.isfinite(energies)) else float("nan")
    mean_energy = float(np.mean(energies)) if len(energies) and np.all(np.isfinite(energies)) else float("nan")
    q_endpoint = q_path[-1] if len(q_path) else P["q_i"]
    endpoint_delta = q_endpoint - P["q_f"]
    endpoint_error = float(np.linalg.norm(endpoint_delta))
    Gf = G(P["q_f"])
    metric_endpoint_error = float(np.sqrt(max(float(endpoint_delta.T @ Gf @ endpoint_delta), 0.0)))
    conds = np.asarray(conds, dtype=float)
    mean_cond = float(np.mean(conds)) if len(conds) else float("nan")
    max_cond = float(np.max(conds)) if len(conds) else float("nan")
    status = P.get("tiny_segment_status", "normal")
    log_success = bool(info.get("success", False))
    approximate = bool(info.get("approximate", False)) or status == "local_approximation"
    failed = not log_success and model_name == "Full Riemannian M(q)"
    P.update({"energy_values": energies, "energy_drift": energy_drift, "endpoint_error": endpoint_error, "metric_endpoint_error": metric_endpoint_error, "mean_cond_M": mean_cond, "max_cond_M": max_cond, "num_metric_regularizations": int(num_regularized), "max_metric_regularization": float(max_regularization)})
    return {"model": model_name, "segment_index": int(segment_index), "start_idx": int(P["start"]), "end_idx": int(P["end"]), "duration": float(max((P["end"] - P["start"]) * dt, dt)), "num_samples": int(P["end"] - P["start"] + 1), "euclidean_length": euclidean_path_length(q_path), "riemannian_length": float(P.get("length", 0.0)), "endpoint_error": endpoint_error, "metric_endpoint_error": metric_endpoint_error, "energy_drift": energy_drift, "mean_energy": mean_energy, "mean_cond_M": mean_cond, "max_cond_M": max_cond, "num_metric_regularizations": int(num_regularized), "max_metric_regularization": float(max_regularization), "log_map_status": str(info.get("method", "unknown")), "log_map_success": log_success, "failed_log_map": bool(failed), "approx_log_map": bool(approximate), "tiny_segment_status": status, "warning_energy_drift": bool(np.isfinite(energy_drift) and energy_drift > max_allowed_energy_drift), "warning_cond_M": bool(np.isfinite(max_cond) and max_cond > max_allowed_cond_M)}


## 8. Joint-Space Validation

The reported errors are intentionally narrow.

Ordinary joint reconstruction error:

$$
\mathrm{joint\_mse}=\frac{1}{T}\sum_{t=1}^{T}\|q_t-\hat{q}_t\|_2^2.
$$

Metric-weighted reconstruction error:

$$
\mathrm{metric\_joint\_mse}=\frac{1}{T}\sum_{t=1}^{T}(q_t-\hat{q}_t)^\top M(q_t)(q_t-\hat{q}_t).
$$

Metric-speed profile error:

$$
\mathrm{velocity\_profile\_mse}=\frac{1}{T}\sum_{t=1}^{T}\left(\|\dot{q}_t\|_{q_t}-\|\dot{\hat{q}}_t\|_{\hat{q}_t}\right)^2.
$$

Endpoint, energy, and metric-conditioning diagnostics are recorded per primitive.


In [ ]:
# --------------------
# Validation
# --------------------
def joint_errors(q: np.ndarray, q_hat: np.ndarray, G: Metric) -> tuple[float, float]:
    n = min(len(q), len(q_hat))
    if n == 0:
        return float("nan"), float("nan")

    q = q[:n]
    q_hat = q_hat[:n]
    euclidean = float(np.mean(np.sum((q - q_hat) ** 2, axis=1)))
    metric = float(np.mean([(q[t] - q_hat[t]).T @ G(q[t]) @ (q[t] - q_hat[t]) for t in range(n)]))
    return euclidean, metric


def velocity_profile_error(q: np.ndarray, qdot: np.ndarray, q_hat: np.ndarray, dt: float, G: Metric) -> float:
    if len(q_hat) < 2:
        return float("nan")

    qdot_hat = finite_difference_velocity(q_hat, dt)
    n = min(len(q), len(q_hat), len(qdot), len(qdot_hat))
    v = np.array([norm(q[t], qdot[t], G) for t in range(n)])
    v_hat = np.array([norm(q_hat[t], qdot_hat[t], G) for t in range(n)])
    return float(np.mean((v - v_hat) ** 2))


def all_errors(q: np.ndarray, qdot: np.ndarray, q_hat: np.ndarray, dt: float, metric_for_joint_error: Metric, metric_for_speed: Metric | None = None) -> dict:
    """Return errors under a common reference metric and, optionally, a native speed metric.

    `metric_joint_mse` and `velocity_profile_mse` are the fair cross-model values:
    both are computed under `metric_for_joint_error`, normally the full human M(q).
    `velocity_profile_mse_native` is retained only as a model-internal diagnostic.
    """
    joint_mse, metric_joint_mse = joint_errors(q, q_hat, metric_for_joint_error)
    speed_ref = velocity_profile_error(q, qdot, q_hat, dt, metric_for_joint_error)
    speed_native = speed_ref if metric_for_speed is None else velocity_profile_error(q, qdot, q_hat, dt, metric_for_speed)
    return {
        "joint_mse": joint_mse,
        "metric_joint_mse": metric_joint_mse,
        "velocity_profile_mse": speed_ref,
        "velocity_profile_mse_reference": speed_ref,
        "velocity_profile_mse_native": speed_native,
    }


## 9. Per-Trial Pipeline

For each repetition, the notebook executes the same procedure:

1. Convert angles to radians and interpolate small gaps.
2. Detect movement onset and keep the reach-return interval.
3. Smooth joint angles and estimate $q(t)$ and $\dot{q}(t)$.
4. Compute Riemannian and Euclidean segment boundaries.
5. Reconstruct each requested model as $\hat{q}(t)$.
6. Store trial-level errors and per-segment diagnostics.


In [ ]:
# --------------------
# Per-trial processing
# --------------------
def interpolate_nans_columnwise(q: np.ndarray) -> np.ndarray:
    q = np.asarray(q, dtype=float).copy()
    sample_index = np.arange(q.shape[0])
    for j in range(q.shape[1]):
        y = q[:, j]
        good = np.isfinite(y)
        if np.sum(good) < 2:
            raise ValueError(f"Joint {j} has too few valid samples for interpolation.")
        q[:, j] = np.interp(sample_index, sample_index[good], y[good])
    return q


def summarize_segment_diagnostics(segment_rows: list[dict]) -> dict:
    def finite_values(name: str) -> list[float]:
        return [float(row[name]) for row in segment_rows if name in row and np.isfinite(float(row[name]))]
    endpoint = finite_values("endpoint_error")
    metric_endpoint = finite_values("metric_endpoint_error")
    energy = finite_values("energy_drift")
    conds = finite_values("max_cond_M")
    return {"mean_endpoint_error": float(np.mean(endpoint)) if endpoint else float("nan"), "max_endpoint_error": float(np.max(endpoint)) if endpoint else float("nan"), "mean_metric_endpoint_error": float(np.mean(metric_endpoint)) if metric_endpoint else float("nan"), "mean_energy_drift": float(np.mean(energy)) if energy else float("nan"), "max_energy_drift": float(np.max(energy)) if energy else float("nan"), "mean_cond_M": float(np.mean(conds)) if conds else float("nan"), "max_cond_M": float(np.max(conds)) if conds else float("nan"), "num_metric_regularizations": int(sum(int(row.get("num_metric_regularizations", 0)) for row in segment_rows)), "max_metric_regularization": float(max([float(row.get("max_metric_regularization", 0.0)) for row in segment_rows] or [0.0])), "num_failed_log_maps": int(sum(bool(row.get("failed_log_map", False)) for row in segment_rows)), "num_approx_log_maps": int(sum(bool(row.get("approx_log_map", False)) for row in segment_rows))}


def preprocess_trial_motion(trial: ReachGraspTrial, G: Metric, min_reach_duration: float, savgol_window_samples: int = 21, savgol_polyorder: int = 2) -> dict:
    if len(trial.time) < 2:
        raise ValueError("Trial has fewer than two samples.")
    dt = float(np.median(np.diff(trial.time)))
    q_raw = np.asarray(trial.arm_joint_angles, dtype=float)
    if trial.angles_in_degrees:
        q_raw = np.deg2rad(q_raw)
    missing_fraction = float(np.mean(np.isnan(q_raw)))
    if missing_fraction > 0.10:
        raise ValueError(f"More than 10% of selected arm joint samples are missing: {missing_fraction:.3f}")
    if np.any(np.isnan(q_raw)):
        q_raw = interpolate_nans_columnwise(q_raw)
    hand_motion = np.asarray(trial.hand_motion, dtype=float)
    if trial.angles_in_degrees:
        hand_motion = np.deg2rad(hand_motion)
    if len(hand_motion) != len(q_raw):
        raise ValueError("Hand motion and arm joint arrays must have the same number of samples.")
    if np.any(np.isnan(hand_motion)):
        hand_motion = interpolate_nans_columnwise(hand_motion)
    t_start = detect_movement_onset(hand_motion, dt)
    t_action_end = len(q_raw) - 1
    q_reach_raw = q_raw[t_start:t_action_end + 1]
    hand_reach = hand_motion[t_start:t_action_end + 1]
    reach_duration = (len(q_reach_raw) - 1) * dt
    if reach_duration < min_reach_duration:
        raise ValueError(f"Reach phase is too short: {reach_duration:.3f} s")
    t_reach = np.arange(len(q_reach_raw), dtype=float) * dt
    q, qdot = smooth_and_differentiate(
        q_reach_raw, t_reach, window_length=savgol_window_samples, polyorder=savgol_polyorder)
    active_start = first_meaningful_velocity_index(q, qdot, G)
    if active_start > 0:
        q = q[active_start:]
        qdot = qdot[active_start:]
        hand_reach = hand_reach[active_start:]
        t_start += active_start
        reach_duration = (len(q) - 1) * dt
    return {"dt": dt, "q": q, "qdot": qdot, "hand_reach": hand_reach, "t_start": t_start, "t_action_end": t_action_end, "active_start": active_start, "reach_duration": reach_duration}


def build_reconstruction_row(common: dict, model_name: str, segmentation_name: str, spatial_method: str, q: np.ndarray, qdot: np.ndarray, q_hat: np.ndarray, dt: float, G_reference: Metric, speed_metric: Metric, primitives: list[dict], model_segment_rows: list[dict], max_metric_joint_mse: float, max_velocity_profile_mse: float, max_allowed_energy_drift: float, max_allowed_cond_M: float) -> dict:
    errors = all_errors(q, qdot, q_hat, dt, G_reference, speed_metric)
    diagnostics_summary = summarize_segment_diagnostics(model_segment_rows)
    rejection_reasons = []
    if len(primitives) < 1:
        rejection_reasons.append("no_primitives")
    if len(primitives) > CONFIG["max_segments"]:
        rejection_reasons.append("too_many_primitives")
    if not np.isfinite(errors["metric_joint_mse"]):
        rejection_reasons.append("metric_joint_mse_nonfinite")
    elif errors["metric_joint_mse"] > max_metric_joint_mse:
        rejection_reasons.append("metric_joint_mse_too_high")
    if not np.isfinite(errors["velocity_profile_mse"]):
        rejection_reasons.append("velocity_profile_mse_nonfinite")
    elif errors["velocity_profile_mse"] > max_velocity_profile_mse:
        rejection_reasons.append("velocity_profile_mse_too_high")
    if diagnostics_summary["num_failed_log_maps"] > 0:
        rejection_reasons.append("log_map_not_all_converged")
    if np.isfinite(diagnostics_summary["max_energy_drift"]) and diagnostics_summary["max_energy_drift"] > max_allowed_energy_drift:
        rejection_reasons.append("energy_drift_warning")
    if np.isfinite(diagnostics_summary["max_cond_M"]) and diagnostics_summary["max_cond_M"] > max_allowed_cond_M:
        rejection_reasons.append("metric_condition_warning")
    return {**common, "model": model_name, "segmentation": segmentation_name, "metric_type": model_name, "spatial_method": spatial_method, "num_primitives": int(len(primitives)), **errors, **diagnostics_summary, "accepted": bool(len(rejection_reasons) == 0), "rejection_reasons": ";".join(rejection_reasons)}


def process_reach_grasp_trial(
    trial: ReachGraspTrial,
    G: Metric,
    delta_theta: float = np.pi / 6,
    angle_confirm_samples: int = 5,
    min_segment_samples: int = 10,
    speed_floor_fraction: float = 0.05,
    tiny_segment_policy: str = "merge",
    min_tiny_segment_duration: float = 0.15,
    min_reach_duration: float = 0.30,
    min_hand_speed: float = 1e-4,
    max_metric_joint_mse: float = 0.05,
    max_velocity_profile_mse: float = 0.50,
    geodesic_path_samples: int = 80,
    log_map_max_nfev: int = 60,
    log_map_endpoint_tol: float = 2e-3,
    geodesic_max_step: float = 0.03,
    max_allowed_energy_drift: float = 0.05,
    max_allowed_cond_M: float = 1e8,
    constant_metric_stride: int = 5,
    savgol_window_samples: int = 21,
    savgol_polyorder: int = 2,
) -> tuple[list[dict], list[dict], list[dict]]:
    data = preprocess_trial_motion(trial, G, min_reach_duration=min_reach_duration, savgol_window_samples=savgol_window_samples, savgol_polyorder=savgol_polyorder)
    q, qdot, dt = data["q"], data["qdot"], data["dt"]

    hand_speed = np.linalg.norm(finite_difference_velocity(data["hand_reach"], dt), axis=1)
    max_hand_speed = float(np.max(hand_speed)) if len(hand_speed) > 0 else 0.0
    if max_hand_speed < min_hand_speed:
        raise ValueError("Detected hand/wrist movement speed is below threshold.")

    speeds = metric_speeds(q, qdot, G)
    speed_floor = max(1e-10, float(speed_floor_fraction) * float(np.nanmax(speeds))) if len(speeds) else 1e-10
    riem_segments_raw = segment_riemannian(
        q, qdot, G, delta_theta=delta_theta, min_segment_samples=min_segment_samples,
        speed_floor_fraction=speed_floor_fraction, angle_confirm_samples=angle_confirm_samples)
    riem_segments, riem_local_segments, tiny_counts, riem_segment_status = handle_tiny_segments(riem_segments_raw, q, dt, policy=tiny_segment_policy, min_duration=min_tiny_segment_duration)

    eucl_segments = segment_euclidean_zero_velocity_crossing(qdot, dt, window_s=CONFIG["euclidean_zero_crossing_window_s"], min_joints=CONFIG["euclidean_zero_crossing_min_joints"], min_segment_samples=min_segment_samples)
    common = {
        "subject": trial.subject, "task": trial.task, "trial": trial.trial, "trial_id": f"{trial.subject}_{trial.task}_trial-{trial.trial:02d}",
        "t_start": int(data["t_start"]), "t_action_end": int(data["t_action_end"]), "active_start_offset": int(data["active_start"]),
        "reach_duration": float(data["reach_duration"]), "riemannian_num_segments_raw": int(len(riem_segments_raw)),
        "riemannian_num_segments_after_tiny": int(len(riem_segments)), "euclidean_num_segments": int(len(eucl_segments)),
        "tiny_segments_detected": int(tiny_counts.get("tiny_segments_detected", 0)), "tiny_segments_merged": int(tiny_counts.get("tiny_segments_merged", 0)),
        "tiny_segments_local": int(tiny_counts.get("tiny_segments_local", 0)), "tiny_segments_shot": int(tiny_counts.get("tiny_segments_shot", 0)),
        "max_hand_speed": max_hand_speed, "speed_floor": float(speed_floor),
    }

    requested_models = set(CONFIG.get("models_to_run", ["riemannian", "euclidean_paper", "euclidean_controlled", "constant_metric"]))
    valid_models = {"riemannian", "euclidean_paper", "euclidean_controlled", "constant_metric"}
    unknown_models = sorted(requested_models - valid_models)
    if unknown_models:
        raise ValueError(f"Unknown model keys in CONFIG['models_to_run']: {unknown_models}")

    models = []
    if "riemannian" in requested_models:
        models.append(("Full Riemannian M(q)", "riemannian_parallel_transport", G, "geodesic", riem_segments, riem_local_segments, riem_segment_status))
    if "euclidean_paper" in requested_models:
        models.append(("Euclidean paper baseline", "euclidean_zero_velocity", EuclideanMetricModel(q.shape[1]), "straight", eucl_segments, set(), {}))
    if "euclidean_controlled" in requested_models:
        models.append(("Euclidean controlled baseline", "riemannian_parallel_transport", EuclideanMetricModel(q.shape[1]), "straight", riem_segments, set(), riem_segment_status))
    if "constant_metric" in requested_models:
        constant_G = constant_metric_from_trial(q, G, stride=constant_metric_stride)
        models.append(("Constant metric controlled baseline", "riemannian_parallel_transport", constant_G, "straight", riem_segments, set(), riem_segment_status))

    summary_rows, segment_rows, model_results = [], [], []
    for model_name, segmentation_name, model_metric, spatial_method, model_segments, local_segments, segment_status in models:
        if spatial_method == "straight":
            primitives = build_primitives_straight(q, model_segments, model_metric, path_samples=geodesic_path_samples, segment_status=segment_status)
        else:
            primitives = build_primitives_riemannian(q, model_segments, model_metric, path_samples=geodesic_path_samples, log_map_max_nfev=log_map_max_nfev, endpoint_tol=log_map_endpoint_tol, max_step=geodesic_max_step, local_segments=local_segments, segment_status=segment_status)

        primitives = add_temporal_profiles(primitives, q, qdot, dt, model_metric)
        primitives = smooth_boundary_speeds(primitives)
        q_hat = reconstruct_all_riemannian(primitives, len(q), dt)
        model_segment_rows = [
            primitive_diagnostics(P, model_metric, dt, model_name, segment_index, max_allowed_energy_drift, max_allowed_cond_M)
            for segment_index, P in enumerate(primitives)
        ]
        for row in model_segment_rows:
            row.update({"trial_id": common["trial_id"], "subject": trial.subject, "task": trial.task, "trial": trial.trial, "model": model_name, "segmentation": segmentation_name})

        summary = build_reconstruction_row(common, model_name, segmentation_name, spatial_method, q, qdot, q_hat, dt, G, model_metric, primitives, model_segment_rows, max_metric_joint_mse, max_velocity_profile_mse, max_allowed_energy_drift, max_allowed_cond_M)
        summary_rows.append(summary)
        segment_rows.extend(model_segment_rows)
        model_results.append({"trial": trial, "model": model_name, "metric": model_metric, "primitives": primitives, "q": q, "qdot": qdot, "q_hat": q_hat, "dt": dt, "summary": summary, "segment_rows": model_segment_rows})

    return summary_rows, segment_rows, model_results


## 10. Save Tables and Plot Diagnostics

The saved tables are the durable result of the run. The plots are sanity checks for shifted reconstructions, strange speed profiles, and suspicious segment boundaries.


In [ ]:
# --------------------
# Output helpers
# --------------------
def safe_model_name(name: str) -> str:
    return "".join(ch.lower() if ch.isalnum() else "_" for ch in name).strip("_")


def plot_trial(result: dict) -> None:
    if plt is None:
        print("Matplotlib is not installed; skipping diagnostic plot.")
        return
    trial = result["trial"]
    primitives = result["primitives"]
    q = result["q"]
    qdot = result["qdot"]
    q_hat = result["q_hat"]
    G = result["metric"]
    model_name = result["model"]
    n = min(len(q), len(q_hat))
    t = np.arange(n)
    dt = result["dt"]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(t, q[:n], alpha=0.7)
    ax.plot(t, q_hat[:n], linestyle="--", alpha=0.7, label="reconstructed")
    ax.set_title(f"Joint trajectories: {model_name}")
    ax.set_xlabel("sample")
    ax.legend()
    plt.show()
    qdot_hat = finite_difference_velocity(q_hat[:n], dt) if len(q_hat) else np.zeros_like(q[:n])
    speed = np.array([norm(q[i], qdot[i], G) for i in range(n)])
    speed_hat = np.array([norm(q_hat[i], qdot_hat[i], G) for i in range(n)]) if len(q_hat) else np.full(n, np.nan)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(t, speed, label="original")
    ax.plot(t, speed_hat, label="reconstructed")
    ax.set_title(f"Metric speed: {model_name}")
    ax.set_xlabel("sample")
    ax.legend()
    plt.show()
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(t, speed, color="black")
    for P in primitives:
        ax.axvline(P["start"], color="tab:blue", alpha=0.5)
        ax.axvline(P["end"], color="tab:red", alpha=0.3)
    ax.set_title(f"Segment boundaries: {model_name}")
    ax.set_xlabel("sample")
    plt.show()


def save_trial_plots(output_dir: Path, result: dict) -> None:
    if plt is None:
        return
    output_dir.mkdir(parents=True, exist_ok=True)
    trial = result["trial"]
    q = result["q"]
    q_hat = result["q_hat"]
    model_name = result["model"]
    prefix = f"{trial.subject}_{trial.task}_trial-{trial.trial:02d}_{safe_model_name(model_name)}"
    n = min(len(q), len(q_hat))
    t = np.arange(n)
    fig, ax = plt.subplots()
    ax.plot(t, q[:n], alpha=0.7, label="original")
    ax.plot(t, q_hat[:n], linestyle="--", alpha=0.7, label="reconstructed")
    ax.set_title(f"Joint trajectories: {model_name}")
    ax.set_xlabel("sample")
    ax.legend()
    fig.savefig(output_dir / f"{prefix}_joint_trajectories.png", dpi=160, bbox_inches="tight")
    plt.close(fig)


def write_summary_table(rows: list[dict], output_path: Path) -> None:
    fieldnames = [
        "subject", "task", "trial", "trial_id", "model", "segmentation", "metric_type", "spatial_method",
        "t_start", "t_action_end", "active_start_offset", "reach_duration", "riemannian_num_segments_raw",
        "riemannian_num_segments_after_tiny", "euclidean_num_segments", "tiny_segments_detected", "tiny_segments_merged",
        "tiny_segments_local", "tiny_segments_shot", "num_primitives", "joint_mse", "metric_joint_mse",
        "velocity_profile_mse", "velocity_profile_mse_reference", "velocity_profile_mse_native", "max_hand_speed", "speed_floor",
        "mean_endpoint_error", "max_endpoint_error", "mean_metric_endpoint_error", "num_failed_log_maps",
        "num_approx_log_maps", "mean_energy_drift", "max_energy_drift", "mean_cond_M", "max_cond_M",
        "num_metric_regularizations", "max_metric_regularization", "accepted", "rejection_reasons", "error",
    ]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({name: row.get(name, "") for name in fieldnames})


def write_segment_table(rows: list[dict], output_path: Path) -> None:
    fieldnames = ["trial_id", "subject", "task", "trial", "model", "segmentation", "segment_index", "start_idx", "end_idx", "duration", "num_samples", "euclidean_length", "riemannian_length", "endpoint_error", "metric_endpoint_error", "energy_drift", "mean_energy", "mean_cond_M", "max_cond_M", "num_metric_regularizations", "max_metric_regularization", "log_map_status", "log_map_success", "failed_log_map", "approx_log_map", "tiny_segment_status", "warning_energy_drift", "warning_cond_M"]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({name: row.get(name, "") for name in fieldnames})


def primitive_record(result: dict, segment_index: int, P: dict) -> dict:
    trial = result["trial"]
    return {
        "subject": trial.subject,
        "task": trial.task,
        "trial": int(trial.trial),
        "trial_id": f"{trial.subject}_{trial.task}_trial-{trial.trial:02d}",
        "model": result["model"],
        "segment_index": int(segment_index),
        "start_idx": int(P["start"]),
        "end_idx": int(P["end"]),
        "duration": float(P.get("duration", np.nan)),
        "length": float(P.get("length", np.nan)),
        "speed0": float(P.get("speed0", np.nan)),
        "speedf": float(P.get("speedf", np.nan)),
        "tiny_segment_status": str(P.get("tiny_segment_status", "normal")),
        "log_map_success": bool(P.get("log_map_info", {}).get("success", False)),
        "q_i": np.asarray(P["q_i"], dtype=float),
        "q_f": np.asarray(P["q_f"], dtype=float),
        "v_i": np.asarray(P["v_i"], dtype=float),
        "arc_phase_coeffs": np.asarray(P.get("arc_phase_coeffs", []), dtype=float),
        "q_path": np.asarray(P.get("q_path", []), dtype=float),
        "arc_lengths": np.asarray(P.get("arc_lengths", []), dtype=float),
    }


def save_primitive_library(results: list[dict], output_dir: Path, models: list[str] | None = None) -> tuple[Path, Path | None]:
    selected = set(models or [])
    records = []
    metadata = []
    for result in results:
        if selected and result["model"] not in selected:
            continue
        for segment_index, P in enumerate(result["primitives"]):
            rec = primitive_record(result, segment_index, P)
            records.append(rec)
            metadata.append({k: v for k, v in rec.items() if not isinstance(v, np.ndarray)})

    output_dir.mkdir(parents=True, exist_ok=True)
    library_path = output_dir / "human_geodesic_primitive_library.npz"
    np.savez_compressed(library_path, records=np.asarray(records, dtype=object))

    metadata_path = output_dir / "human_geodesic_primitive_library_metadata.csv"
    if pd is not None:
        pd.DataFrame(metadata).to_csv(metadata_path, index=False)
    else:
        if metadata:
            with metadata_path.open("w", newline="", encoding="utf-8") as handle:
                writer = csv.DictWriter(handle, fieldnames=list(metadata[0].keys()))
                writer.writeheader()
                writer.writerows(metadata)
        else:
            metadata_path = None
    return library_path, metadata_path


def plot_k_distribution(rows: list[dict]) -> None:
    if plt is None:
        print("Matplotlib is not installed; skipping primitive-count plot.")
        return
    good_rows = [row for row in rows if "num_primitives" in row and "model" in row]
    if not good_rows:
        print("No primitive counts to plot.")
        return
    fig, ax = plt.subplots(figsize=(8, 4))
    for model in sorted({row["model"] for row in good_rows}):
        ks = [row["num_primitives"] for row in good_rows if row["model"] == model and isinstance(row.get("num_primitives"), (int, float))]
        if not ks:
            continue
        bins = np.arange(0.5, max(ks) + 1.5, 1.0)
        ax.hist(ks, bins=bins, alpha=0.45, label=model)
    ax.set_xlabel("number of primitives")
    ax.set_ylabel("trial/model rows")
    ax.set_title("Primitive-count distribution")
    ax.legend()
    plt.show()


## 11. Run the Experiment

This cell processes every configured subject, task, and repetition. Failures are recorded as rows so a bad trial does not hide the rest of the experiment.


In [ ]:
rows = []
segment_rows = []
trial_results = []
G = MassMetricModel(finite_difference_step=CONFIG["metric_finite_difference_step"], eig_floor=CONFIG["metric_eig_floor"], name="Full Riemannian M(q)")

trial_kwargs = {
    "delta_theta": CONFIG["delta_theta"],
    "angle_confirm_samples": CONFIG["angle_confirm_samples"],
    "min_segment_samples": CONFIG["min_segment_samples"],
    "speed_floor_fraction": CONFIG["speed_floor_fraction"],
    "tiny_segment_policy": CONFIG["tiny_segment_policy"],
    "min_tiny_segment_duration": CONFIG["min_tiny_segment_duration"],
    "min_reach_duration": CONFIG["min_reach_duration"],
    "min_hand_speed": CONFIG["min_hand_speed"],
    "max_metric_joint_mse": CONFIG["max_metric_joint_mse"],
    "max_velocity_profile_mse": CONFIG["max_velocity_profile_mse"],
    "geodesic_path_samples": CONFIG["geodesic_path_samples"],
    "log_map_max_nfev": CONFIG["log_map_max_nfev"],
    "log_map_endpoint_tol": CONFIG["log_map_endpoint_tol"],
    "geodesic_max_step": CONFIG["geodesic_max_step"],
    "max_allowed_energy_drift": CONFIG["max_allowed_energy_drift"],
    "max_allowed_cond_M": CONFIG["max_allowed_cond_M"],
    "constant_metric_stride": CONFIG["constant_metric_stride"],
    "savgol_window_samples": CONFIG["savgol_window_samples"],
    "savgol_polyorder": CONFIG["savgol_polyorder"],
}

for subject in CONFIG["subjects"]:
    for task in CONFIG["tasks"]:
        print(f"\n>>> Processing {subject} {task} <<<")
        try:
            trials = load_randg_trials(subject, task, dataset_path=CONFIG["dataset_path"])
        except Exception as exc:
            print(f"Could not load {subject} {task}: {exc}")
            continue

        for trial in tqdm(trials, desc=f"{subject} {task}", unit="trial", leave=True):
            try:
                trial_summary_rows, trial_segment_rows, trial_model_results = process_reach_grasp_trial(trial, G=G, **trial_kwargs)
                rows.extend(trial_summary_rows)
                segment_rows.extend(trial_segment_rows)
                trial_results.extend(trial_model_results)
            except Exception as exc:
                print(f"Error processing trial {trial.trial}: {exc}")
                rows.append({"subject": trial.subject, "task": trial.task, "trial": trial.trial, "trial_id": f"{trial.subject}_{trial.task}_trial-{trial.trial:02d}", "model": "", "metric_type": "", "accepted": False, "rejection_reasons": "exception", "error": repr(exc)})

summary_path = CONFIG["output_dir_full"] / "riemannian_geodesic_results.csv"
segment_path = CONFIG["output_dir_full"] / "riemannian_geodesic_segment_diagnostics.csv"
if pd is not None:
    df = pd.DataFrame(rows)
    seg_df = pd.DataFrame(segment_rows)
    df.to_csv(summary_path, index=False)
    seg_df.to_csv(segment_path, index=False)
    print(f"\nDone. Saved summary results to {summary_path}")
    print(f"Saved per-segment diagnostics to {segment_path}")
    if not df.empty:
        cols = ["model", "segmentation", "num_primitives", "joint_mse", "metric_joint_mse", "velocity_profile_mse", "velocity_profile_mse_native", "mean_endpoint_error", "mean_energy_drift"]
        cols = [c for c in cols if c in df.columns]
        print("\n--- MEAN MODEL COMPARISON SUMMARY ---")
        print(df[cols].groupby(["model", "segmentation"], dropna=False).mean(numeric_only=True))
else:
    write_summary_table(rows, summary_path)
    write_segment_table(segment_rows, segment_path)
    print(f"\nDone. Saved summary results to {summary_path}")
    print(f"Saved per-segment diagnostics to {segment_path}")

primitive_library_path, primitive_metadata_path = save_primitive_library(trial_results, CONFIG["output_dir_full"], models=CONFIG.get("primitive_library_models"))
print(f"Saved primitive library to {primitive_library_path}")
if primitive_metadata_path is not None:
    print(f"Saved primitive metadata to {primitive_metadata_path}")


## 12. Summary Table

The main comparison columns are `model`, `segmentation`, `spatial_method`, `num_primitives`, `joint_mse`, `metric_joint_mse`, `velocity_profile_mse`, endpoint diagnostics, energy drift, and metric-conditioning diagnostics.

The central comparison is whether the Riemannian reconstruction lowers the metric-aware errors under $M(q)$ while keeping the primitive count and diagnostics reasonable.


In [ ]:
if pd is not None:
    display(pd.DataFrame(rows))
else:
    rows[:5]

## 13. Diagnostic Plots

Use these plots before trusting the summary metrics. They show joint reconstruction, metric-speed profiles, and segment boundaries for the first configured trials.


In [ ]:
for result in trial_results[:CONFIG["plot_first_n_trials"]]:
    plot_trial(result)
    save_trial_plots(CONFIG["output_dir_full"] / "plots", result)

plot_k_distribution(rows)


In [ ]:
rows = []

for result in trial_results:
    for i, (A, B) in enumerate(zip(result["primitives"][:-1], result["primitives"][1:])):
        rows.append({
            "model": result["model"],
            "trial_id": result["summary"]["trial_id"],
            "boundary": i,
            "A_end": A["end"],
            "B_start": B["start"],
            "index_gap": B["start"] - A["end"],
            "position_jump": np.linalg.norm(A["q_path"][-1] - B["q_path"][0]),
            "velocity_jump": np.linalg.norm(A["v_path"][-1] - B["v_path"][0]),
            "speed_jump": abs(A["speedf"] - B["speed0"]),
            "A_status": A["tiny_segment_status"],
            "B_status": B["tiny_segment_status"],
        })

gap_df = pd.DataFrame(rows)
display(gap_df.sort_values("speed_jump", ascending=False))